# Differential Conservation of GPCR Contacts - A Case Study

A great - and inspirational - example is provided by the following paper: https://www.nature.com/articles/nature19107

In it, the authors study shared and differential conservation of atomic contacts between GPCRs in the active and inactive states. The paper provides a great example of the usefulness of native contacts in protein, and the utility of being able to study and compare contacts across multiple structures, states, and proteins. In the following notebook, we will reproduce some of the results of this paper using `Lahuta` and in the process highlight the ease with which one can study contacts in proteins using `Lahuta`.

## 1. First, let's download the structures

In [2]:
from lahuta.api import download_structures

In [3]:
inactive_gpcrs = [
    "1GZM", "2Z73", "2VT4", "2RH1", "3PBL", "3RZE", "3UON", 
    "4DAJ", "3ODU", "4MBS", "4DJH", "4DKL", "4EA3", "4EJ4", 
    "4S0V", "4YAY", "3VW7", "3V2Y", "4Z36", "3EML", "4XNV"
]

# "4ZWJ" has issues, so we're not using it
active_gpcrs = [
    "3PQR", "2YDV", "3SN6", "4MQS", "5C1M", "4XT1"
]

In [4]:
# `data` contains the downloaded structures: pdb_id: paht/to/pdb_file
data = download_structures([*inactive_gpcrs, *active_gpcrs], dir_loc='data')

## 2. Let's get the sequences of the downloaded structures

We define a function that we provide to the file processor to get the sequences of the structures.

In [5]:
from lahuta import Luni
from lahuta.api import CachedFileProcessor

In [6]:
def process_sequence(file_path: str) -> str:
    """Extracts the sequence from a PDB file."""
    luni = Luni(file_path)
    return luni.sequence

sequence_processor = CachedFileProcessor(
    file_list=list(data.values()),
    worker=process_sequence,
)
sequence_processor.process()

In [7]:
# view the sequences
sequence_processor.results

{'1gzm.pdb': 'XMNGTEGPNFYVPFSNKTGVVRSPFEAPQYYLAEPWQFSMLAAYMFLLIMLGFPINFLTLYVTVQHKKLRTPLNYILLNLAVADLFMVFGGFTTTLYTSLHGYFVFGPTGCNLEGFFATLGGEIALWSLVVLAIERYVVVCKPMSNFRFGENHAIMGVAFTWVMALACAAPPLVGWSRYIPEGMQCSCGIDYYTPHEETNNESFVIYMFVVHFIIPLIVIFFCYGQLVFTVKEAAAQQQESATTQKAEKEVTRMVIIMVIAFLICWLPYAGVAFYIFTHQGSDFGPIFMTIPAFFAKTSAVYNPVIYIMMNKQFRNCMVTTLCCGKNDDEXMNGTEGPNFYVPFSNKTGVVRSPFEAPQYYLAEPWQFSMLAAYMFLLIMLGFPINFLTLYVTVQHKKLRTPLNYILLNLAVADLFMVFGGFTTTLYTSLHGYFVFGPTGCNLEGFFATLGGEIALWSLVVLAIERYVVVCKPMSNFRFGENHAIMGVAFTWVMALACAAPPLVGWSRYIPEGMQCSCGIDYYTPHEETNNESFVIYMFVVHFIIPLIVIFFCYGQLVFTVKEAAAQQQESATTQKAEKEVTRMVIIMVIAFLICWLPYAGVAFYIFTHQGSDFGPIFMTIPAFFAKTSAVYNPVIYIMMNKQFRNCMVTTLCCGKNDDE',
 '2z73.pdb': 'ETWWYNPSIVVHPHWREFDQVPDAVYYSLGIFIGICGIIGCGGNGIVIYLFTKTKSLQTPANMFIINLAFSDFTFSLVNGFPLMTISCFLKKWIFGFAACKVYGFIGGIFGFMSIMTMAMISIDRYNVIGRPMAASKKMSHRRAFIMIIFVWLWSVLWAIGPIFGWGAYTLEGVLCNCSFDYISRDSTTRSNILCMFILGFFGPILIIFFCYFNIVMSVSNHEKEMAAMAKRLNAKELRKAQAGANAEMRLAKISIVIVSQFLLSWSPYAVVALLAQFGPLEWVTPYAAQLPVMFAKASAIHNPMIYSV

## 3. Let's align these sequences

As explained in previous sections, we can align the sequences by themselves, but the alignment will be more accurate if we use a pre-aligned MSA that's fine-tuned for GPCRs. <br>
Therefore, we'll use the alignment provided by GPCRdb. 

In [8]:
from lahuta.msa import MSAParser

In [9]:
parser = MSAParser(sequences=sequence_processor.results)
ref_parser = MSAParser('data/gcprdb_all_gpcrs.fasta')

aligned_seqs = parser.align(n_jobs=4, ref_alignment=ref_parser.sequences)
aligned_seqs = aligned_seqs - ref_parser # remove sequences from the reference alignment (not strictly necessary)

INFO:root:Alignment completed. Output file located at: /tmp/tmph6hvnz9k


## 4. Let's process all files and create mapped `NeighborPairs` objects

Now we need to compute the neighbors for each structure. <br>

We, therefore, define a function that we provide to the file processor to compute the neighbors for each structure. We'll make sure to use the aligned sequences to map the neighbors, which will allow us to compare the neighbors across structures.

In [10]:
from pathlib import Path
from lahuta.core.labeled_neighbors import LabeledNeighborPairs

RADIUS = 5.0
RES_DIF = 4

def process_neighbors(file_path: str) -> LabeledNeighborPairs:
    """Extracts the neighbors from a PDB file."""
    luni = Luni(file_path)
    ns = luni.compute_neighbors(radius=RADIUS, res_dif=RES_DIF)
    
    basename = Path(file_path).name
    mapped_ns = ns.map(aligned_seqs.sequences[basename])
    mapped_ns = mapped_ns.remove('resnames').remove('names')
    return mapped_ns

In [11]:
processor = CachedFileProcessor(
    file_list=list(data.values()),
    worker=process_neighbors,
)
processor.process()

In [12]:
processor.results

{'1gzm.pdb': <Lahuta LabeledNeighborPairs class containing 2104 pairs>,
 '2z73.pdb': <Lahuta LabeledNeighborPairs class containing 1992 pairs>,
 '2vt4.pdb': <Lahuta LabeledNeighborPairs class containing 2363 pairs>,
 '2rh1.pdb': <Lahuta LabeledNeighborPairs class containing 1259 pairs>,
 '3pbl.pdb': <Lahuta LabeledNeighborPairs class containing 1733 pairs>,
 '3rze.pdb': <Lahuta LabeledNeighborPairs class containing 809 pairs>,
 '3uon.pdb': <Lahuta LabeledNeighborPairs class containing 1044 pairs>,
 '4daj.pdb': <Lahuta LabeledNeighborPairs class containing 3389 pairs>,
 '3odu.pdb': <Lahuta LabeledNeighborPairs class containing 2515 pairs>,
 '4mbs.pdb': <Lahuta LabeledNeighborPairs class containing 1634 pairs>,
 '4djh.pdb': <Lahuta LabeledNeighborPairs class containing 1923 pairs>,
 '4dkl.pdb': <Lahuta LabeledNeighborPairs class containing 1122 pairs>,
 '4ea3.pdb': <Lahuta LabeledNeighborPairs class containing 1270 pairs>,
 '4ej4.pdb': <Lahuta LabeledNeighborPairs class containing 787 pa

## 5. Shared contacts among active GPCRs

For a simple exercise, we will compute the shared contacts between the active-state GPCRs, as done in the paper. Specifically, we will do the following two steps:
1. Compute the shared contacts between active-state GPCRs
2. Test that we find the important 3x46 - 7x53 contact, as discussed in the paper. 

In terms of actual implementation, we will do the following:
1. We get only the active-state GPCRs from the `processor` object we created in the previous step
2. We compute the intersection of contacts between all active-state GPCRs
3. We them `backmap` the values in the intersection to the original indices 
4. We test that we find the 3x46 - 7x53 contact.

#### 5.1 Get only the active GPCRs

In [13]:
active_lnps = [processor.results[f'{x.lower()}.pdb'] for x in active_gpcrs]
active_lnps

[<Lahuta LabeledNeighborPairs class containing 928 pairs>,
 <Lahuta LabeledNeighborPairs class containing 761 pairs>,
 <Lahuta LabeledNeighborPairs class containing 3242 pairs>,
 <Lahuta LabeledNeighborPairs class containing 864 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1669 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1335 pairs>]

#### 5.2 Compute the intersection

In [14]:
from lahuta.api import intersection

In [15]:
isect = intersection(*active_lnps)
isect.pairs[:3]

array([[('', '1581', ''), ('', '1653', '')],
       [('', '1584', ''), ('', '1649', '')],
       [('', '1588', ''), ('', '1644', '')]],
      dtype=[('names', '<U25'), ('resids', '<U25'), ('resnames', '<U25')])

#### 5.3 & 5.4 Backmap and Test the presence of 3x46-7x53 contacts

In [16]:
active_results = {
    '3pqr.pdb': [131, 306],
    '2ydv.pdb': [98,  288],
    '3sn6.pdb': [127, 326],
    '4mqs.pdb': [117, 440],
    '5c1m.pdb': [161, 336],
    '4xt1.pdb': [125, 291]
}

In [17]:
import numpy as np
from lahuta.utils.array_utils import issubset

active_gpcrs_data = {Path(data[key]).name: data[key] for key in active_gpcrs}
for pdb_id, file_path in active_gpcrs_data.items():
    print (f'Working on {pdb_id}. Status: ', end=' ')
    luni = Luni(file_path)
    ns = luni.compute_neighbors(radius=RADIUS, res_dif=RES_DIF)
    ns = ns.backmap(aligned_seqs.sequences[pdb_id], isect.pairs)
    
    assert issubset(np.array([active_results.get(pdb_id)]), ns.resids)
    
    print(f'Passed!')


Working on 3pqr.pdb. Status:  Passed!
Working on 2ydv.pdb. Status:  Passed!
Working on 3sn6.pdb. Status:  Passed!
Working on 4mqs.pdb. Status:  Passed!
Working on 5c1m.pdb. Status:  Passed!
Working on 4xt1.pdb. Status:  Passed!


## Playground

In [18]:
# helper functions
from lahuta.api import count_unique_pairs_across_keys, map_unique_pairs_to_keys

In [19]:
# get only the inactive GPCRs
inactive_lnps = [processor.results[f'{x.lower()}.pdb'] for x in inactive_gpcrs]
inactive_lnps

[<Lahuta LabeledNeighborPairs class containing 2104 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1992 pairs>,
 <Lahuta LabeledNeighborPairs class containing 2363 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1259 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1733 pairs>,
 <Lahuta LabeledNeighborPairs class containing 809 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1044 pairs>,
 <Lahuta LabeledNeighborPairs class containing 3389 pairs>,
 <Lahuta LabeledNeighborPairs class containing 2515 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1634 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1923 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1122 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1270 pairs>,
 <Lahuta LabeledNeighborPairs class containing 787 pairs>,
 <Lahuta LabeledNeighborPairs class containing 1214 pairs>,
 <Lahuta LabeledNeighborPairs class containing 656 pairs>,
 <Lahuta LabeledNeighborPairs class contain

In [20]:
# compute the intersection
inactive_isect = intersection(*inactive_lnps)

In [21]:
import numpy as np

results = {}
inactive_gpcrs_data = {Path(data[key]).name: data[key] for key in inactive_gpcrs}
for pdb_id, file_path in inactive_gpcrs_data.items():
    luni = Luni(file_path)
    ns = luni.compute_neighbors(radius=RADIUS, res_dif=RES_DIF)
    ns = ns.backmap(aligned_seqs.sequences[pdb_id], isect.pairs)
    results[pdb_id] = ns.resids

In [37]:
# view the results
# results

In [27]:
# count_unique_pairs_across_keys(results)

In [29]:
# map_unique_pairs_to_keys(results)

## Important Notes

A few things to note:

- There is a lot of room for performance improvement. We are computing `NeighbPairs` twice above (once for `mapping` and once for `backmapping`). We could avoid this by computing `NeighbPairs` once and then using `NeighbPairs.map` and `NeighbPairs.backmap` to map the pairs.
- We are removing both `names` and `resnames` before computing the intersection. This is important if we was to commpute contacts between "topologically equivalent" residues. This "topological equivalency" in our case is provided by the MSA. 
- We have barely scratched the surface of what we can do. I hope this notebook provides a good starting point for you to use `Lahuta` to explore your own questions.